# Light at Sea Bottom — 10% / 15% / 20% / 30% thresholds

Based on `notebooks/02_light_attenuation.ipynb`, but instead of a single
euphotic (1% light) classification this notebook produces **one dataset per
light-at-seabed threshold**: 10%, 15%, 20% and 30% of surface light.

Purpose: distinguish where the seabed receives enough light to be mapped from
**satellite** KD490 data (shallow / high-light-at-bottom areas) versus where we
must fall back on **modelling** (deeper / low-light-at-bottom areas).

The light depth for a given surface-light fraction `p` is:

$$z_p = \frac{\ln(100 / p)}{K_{d490}}$$

A pixel is classified `1` (seabed gets >= p% light) if `|depth| < z_p`, else `0`.

**Tip:** all imports, parameters and output paths are defined in the *Setup* cell
below. After a kernel restart you can run *Setup* and jump straight to section 5
(land subdivide + vectorize) as long as the threshold GeoTIFFs already exist on
disk — no need to re-run the KD download / TIF-creation flow.

## 0. Setup — imports, parameters and output paths

In [2]:
import os
from pathlib import Path

import copernicusmarine as cm
import geopandas as gpd
import numpy as np
import rasterio as rio
from osgeo import gdal
from rasterio.warp import transform_bounds

import mnk

gdal.UseExceptions()
# Silence benign pyogrio/GDAL "GDALGetMetadata NULL" noise from /vsimem reads.
gdal.PushErrorHandler("CPLQuietErrorHandler")

out_dir = Path(".")

kd_start_year: int = 2020  #: First year of averaging period
kd_end_year: int = 2025    #: Last year of averaging period
kd_month_start: int = 4    #: Start month of growing season (April)
kd_month_end: int = 10     #: End month of growing season (October)

#: Surface-light percentages reaching the seabed to classify.
LIGHT_PERCENTS = [10, 15, 20, 30]

NODATA = -1
CLASS_NAMES = {1: "Satellittkartleggbar", 0: "Trenger modellering"}

dem_url = "/vsicurl/https://storage.googleapis.com/niva-geodata/MarintNaturKart/features/feature_norge_dem50_depth_filled.tif"
land_url = (
    "gs://niva-geodata/MarintNaturKart/aux/"
    "land-mask-norway_FKB-osm-border-dissolved-fullres_v3_25833_20260924.geo.parquet"
)

# --- Derived output paths (deterministic; defined here so later sections can run
#     without re-running the KD download / TIF-creation flow) ---
kd_wgs84_path = str(out_dir / f"KD490_norge_{kd_start_year}-{kd_end_year}_wgs84.tif")
kd_25833_path = str(out_dir / f"KD490_norge_{kd_start_year}-{kd_end_year}_25833.tif")
kd_filled_path = str(out_dir / f"KD490_norge_{kd_start_year}-{kd_end_year}_filled_25833.tif")

raster_paths = {
    pct: str(
        out_dir
        / f"lys-havbunn-{pct}pct-kd{kd_start_year}-{kd_end_year}_norge_2026_25833.tif"
    )
    for pct in LIGHT_PERCENTS
}
gpkg_path = str(
    out_dir / f"lys-havbunn-kd{kd_start_year}-{kd_end_year}_norge_2026_25833.gpkg"
)
raster_paths

{10: 'lys-havbunn-10pct-kd2020-2025_norge_2026_25833.tif',
 15: 'lys-havbunn-15pct-kd2020-2025_norge_2026_25833.tif',
 20: 'lys-havbunn-20pct-kd2020-2025_norge_2026_25833.tif',
 30: 'lys-havbunn-30pct-kd2020-2025_norge_2026_25833.tif'}

## 1. Bounding box from Norway DEM

In [ ]:
ds = gdal.Open(dem_url)
gt = ds.GetGeoTransform()
dem_xsize, dem_ysize = ds.RasterXSize, ds.RasterYSize
dem_left = gt[0]
dem_top = gt[3]
dem_right = dem_left + gt[1] * dem_xsize
dem_bottom = dem_top + gt[5] * dem_ysize
ds = None

lon_min, lat_min, lon_max, lat_max = transform_bounds(
    "EPSG:25833", "EPSG:4326", dem_left, dem_bottom, dem_right, dem_top,
)
lon_min, lat_min = round(lon_min - 0.5, 1), round(lat_min - 0.5, 1)
lon_max, lat_max = round(lon_max + 0.5, 1), round(lat_max + 0.5, 1)

output_bounds = [dem_left, dem_bottom, dem_right, dem_top]
print(f"Norway bbox (WGS84): lon [{lon_min}, {lon_max}], lat [{lat_min}, {lat_max}]")
print(f"DEM grid: {dem_xsize} x {dem_ysize} @ 50 m")

## 2. Download KD490 from CMEMS (Atlantic 1 km + Arctic 4 km)

In [ ]:
start_date = f"{kd_start_year}-{kd_month_start:02d}-01"
end_date = f"{kd_end_year}-{kd_month_end:02d}-31"
OVERLAP_LAT = 62.0

ds_atl = cm.open_dataset(
    dataset_id="cmems_obs-oc_atl_bgc-transp_my_l3-multi-1km_P1D",
    variables=["KD490"],
    minimum_longitude=lon_min, maximum_longitude=lon_max,
    minimum_latitude=lat_min, maximum_latitude=min(lat_max, 66.0),
    start_datetime=start_date, end_datetime=end_date,
)
ds_arc = cm.open_dataset(
    dataset_id="cmems_obs-oc_arc_bgc-transp_my_l3-multi-4km_P1D",
    variables=["KD490"],
    minimum_longitude=lon_min, maximum_longitude=lon_max,
    minimum_latitude=OVERLAP_LAT, maximum_latitude=lat_max,
    start_datetime=start_date, end_datetime=end_date,
)
print(f"Atlantic shape: {ds_atl['KD490'].shape}")
print(f"Arctic shape:   {ds_arc['KD490'].shape}")

## 3. Seasonal mean mosaic, reproject to EPSG:25833 and gap-fill\n\nThis KD490 raster is shared across all light-percentage thresholds.

In [ ]:
kd_merged, lons, lats = mnk.light.merge_kd490_datasets(
    ds_atl, ds_arc,
    lon_range=(lon_min, lon_max),
    lat_range=(lat_min, lat_max),
    month_range=(kd_month_start, kd_month_end),
)
del ds_atl, ds_arc

mnk.light.save_wgs84_geotiff(kd_merged, lons, lats, kd_wgs84_path)
del kd_merged

mnk.light.reproject_to_25833(kd_wgs84_path, kd_25833_path, output_bounds)
mnk.light.fill_kd490(kd_25833_path, kd_filled_path, output_bounds)

## 4. Classify seabed light for each threshold (10% / 15% / 20% / 30%)

For each percentage, `compute_photic_zone` produces an Int16 raster:
`1` = seabed receives at least that fraction of surface light (**satellite-mappable**),
`0` = below the threshold (**needs modelling**), `-1` = nodata.

In [ ]:
for pct in LIGHT_PERCENTS:
    print(f"\n=== {pct}% light at seabed ===")
    mnk.light.compute_photic_zone(
        dem_path=dem_url,
        kd_path=kd_filled_path,
        out_path=raster_paths[pct],
        output_bounds=output_bounds,
        dem_xsize=dem_xsize,
        dem_ysize=dem_ysize,
        light_percent=float(pct),
    )

## 5. Vectorize each threshold to one GeoPackage

The full-res land mask is a few huge polygons (e.g. one mainland polygon), which
makes per-polygon vector `difference()` very slow. We split the land into a grid
of small pieces **once** (next cell) so each sea polygon only clips against nearby
tiles, then subtract land per threshold. Each threshold becomes its own layer in a
single multi-layer GeoPackage.

`1` -> **Satellittkartleggbar** (seabed light >= threshold), `0` -> **Trenger modellering**.

### 5a. Load and subdivide the land mask (run once)

In [4]:
land = gpd.read_parquet("gs://niva-geodata/MarintNaturKart/aux/land-mask-norway_FKB-osm-seaborder-subdivided-fullres_v3_25833_20260924.geo.parquet")

### 5b. Vectorize thresholds and write the multi-layer GeoPackage

In [5]:
if os.path.exists(gpkg_path):
    os.remove(gpkg_path)

for pct, rpath in raster_paths.items():
    print(f"\n=== Vectorizing {pct}% ===")
    with rio.open(rpath) as src:
        arr = src.read(1)
        profile = src.profile.copy()

    padded = mnk.vectorize.pad_nodata(arr, nodata=NODATA)
    del arr
    padded_path = str(out_dir / f"lys-havbunn-{pct}pct_padded.tif")
    with rio.open(padded_path, "w", **profile) as dst:
        dst.write(padded, 1)
    del padded, profile

    gdf = mnk.vectorize.vectorize_raster_to_gdf(
        padded_path, field_name="lys_int", epsg=25833, nodata=NODATA
    )
    gdf["namn"] = gdf["lys_int"].map(CLASS_NAMES)
    gdf = mnk.vectorize.subtract_land(gdf, land)

    # Append this threshold as its own layer in the shared GeoPackage.
    gdf.to_file(gpkg_path, layer=f"lys_{pct}pct", driver="GPKG")
    print(f"Wrote layer lys_{pct}pct: {len(gdf):,} polygons -> {gpkg_path}")

    del gdf
    os.remove(padded_path)

print(f"\nAll thresholds written to {gpkg_path}")


=== Vectorizing 10% ===
Subtracting land from 28,283 of 43,397 polygons...
Wrote layer lys_10pct: 43,382 polygons -> lys-havbunn-kd2020-2025_norge_2026_25833.gpkg

=== Vectorizing 15% ===
Subtracting land from 30,772 of 44,114 polygons...
Wrote layer lys_15pct: 44,091 polygons -> lys-havbunn-kd2020-2025_norge_2026_25833.gpkg

=== Vectorizing 20% ===
Subtracting land from 32,178 of 44,242 polygons...
Wrote layer lys_20pct: 44,213 polygons -> lys-havbunn-kd2020-2025_norge_2026_25833.gpkg

=== Vectorizing 30% ===
Subtracting land from 33,068 of 43,096 polygons...
Wrote layer lys_30pct: 43,054 polygons -> lys-havbunn-kd2020-2025_norge_2026_25833.gpkg

All thresholds written to lys-havbunn-kd2020-2025_norge_2026_25833.gpkg
